<a href="https://colab.research.google.com/github/BushraAl-Sulami/SDA_BootCamp/blob/main/LangChain_wikipedia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">

<h1><center>RAG with Agent Exercise</center></h1>

# Installing Dependencies

### requirements.txt

openai==0.28  
ipykernel==6.20.2  
langchain==0.0.352  
wikipedia==1.4.0  
rank_bm25==0.2.2   
tiktoken==0.5.2
faiss-cpu==1.7.4

In [ ]:
# !pip install -r requirements.txt

In [ ]:
%pip install "langchain-core==1.6.0" "langchain-classic==1.0.8" "langchain-community==0.4.2" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "langgraph==1.2.11" "openai>=2.54,<4" "wikipedia==1.4.0" "faiss-cpu==1.15.0" "rank-bm25==0.2.2" "tavily-python==0.7.12"


In [ ]:
import getpass
import os

openai_api_key = getpass.getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

Enter your OpenAI API Key: ··········


`gpt-3.5-turbo` is used as our LLM in this lab

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Collecting movie introduction data from Wikipedia

In [ ]:
# https://python.langchain.com/docs/integrations/document_loaders/wikipedia
# Module Wikipedia is required to use WikipediaLoader
from langchain_community.document_loaders import WikipediaLoader

help(WikipediaLoader)

Help on class WikipediaLoader in module langchain_community.document_loaders.wikipedia:

class WikipediaLoader(langchain_core.document_loaders.base.BaseLoader)
 |  WikipediaLoader(
 |      query: str,
 |      lang: str = 'en',
 |      load_max_docs: Optional[int] = 25,
 |      load_all_available_meta: Optional[bool] = False,
 |      doc_content_chars_max: Optional[int] = 4000
 |  )
 |
 |  Load from `Wikipedia`.
 |
 |  The hard limit on the length of the query is 300 for now.
 |
 |  Each wiki page represents one Document.
 |
 |  Method resolution order:
 |      WikipediaLoader
 |      langchain_core.document_loaders.base.BaseLoader
 |      abc.ABC
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      query: str,
 |      lang: str = 'en',
 |      load_max_docs: Optional[int] = 25,
 |      load_all_available_meta: Optional[bool] = False,
 |      doc_content_chars_max: Optional[int] = 4000
 |  )
 |      Initializes a new instance of the WikipediaLoader

In [ ]:
# load MVP
import wikipedia

wikipedia.set_user_agent(
    "ProjectRescueHackathon/1.0 (educational notebook)"
)

endgame_wikipedia_docs = []

for topic in ["Minimum viable product", "Software prototyping"]:
    docs = WikipediaLoader(
        query=topic,
        lang="en",
        load_max_docs=1,
        doc_content_chars_max=12000,
    ).load()

    endgame_wikipedia_docs.extend(docs)

    for doc in docs:
        print("Loaded:", doc.metadata.get("title"))

print("Total documents:", len(endgame_wikipedia_docs))

Loaded: Minimum viable product
Loaded: Software prototyping
Total documents: 2


In [ ]:
len(endgame_wikipedia_docs)

2

In [ ]:
endgame_wikipedia_docs[0]

Document(metadata={'title': 'Minimum viable product', 'summary': "A minimum viable product (MVP) is a version of a product with just enough features to be usable by early customers who can then provide feedback for future product development.\nA focus on releasing an MVP means that developers potentially avoid lengthy and (possibly) unnecessary work. Instead, they iterate on working versions and respond to feedback, challenging and validating assumptions about a product's requirements. The term was coined and defined in 2001 by Frank Robinson and then popularized by Steve Blank and Eric Ries. It may also involve carrying out market analysis beforehand. The MVP is analogous to experimentation in the scientific method applied in the context of validating business hypotheses. It is utilized so that prospective entrepreneurs would know whether a given business idea would actually be viable and profitable by testing the assumptions behind a product or business idea. The concept can be used 

# Loading Movie Reviews from a CSV file

In [ ]:
# https://python.langchain.com/docs/integrations/document_loaders/csv
from langchain_community.document_loaders import CSVLoader

help(CSVLoader)

Help on class CSVLoader in module langchain_community.document_loaders.csv_loader:

class CSVLoader(langchain_core.document_loaders.base.BaseLoader)
 |  CSVLoader(
 |      file_path: Union[str, pathlib._local.Path],
 |      source_column: Optional[str] = None,
 |      metadata_columns: Sequence[str] = (),
 |      csv_args: Optional[Dict] = None,
 |      encoding: Optional[str] = None,
 |      autodetect_encoding: bool = False,
 |      *,
 |      content_columns: Sequence[str] = ()
 |  )
 |
 |  Load a `CSV` file into a list of `Document` objects.
 |
 |  Each document represents one row of the CSV file. Every row is converted
 |  into a key/value pair and outputted to a new line in the document's
 |  page_content.
 |
 |  The source for each document loaded from csv is set to the value of the
 |  `file_path` argument for all documents by default.
 |  You can override this by setting the `source_column` argument to the
 |  name of a column in the CSV file.
 |  The source of each document w

In [ ]:
import csv

tasks = [
    ["T01", "Define MVP scope", 1, "", "Choose core features and acceptance criteria."],
    ["T02", "Extract PDF text", 2, "T01", "Read a text-based PDF without OCR."],
    ["T03", "Build document retrieval", 3, "T02", "Split text, embed, and search passages."],
    ["T04", "Generate answers with sources", 2, "T03", "Answer using retrieved passages and cite sources."],
    ["T05", "Create simple interface", 2, "T04", "Add question input and answer display."],
    ["T06", "Add web search", 1, "T04", "Search for recent information."],
    ["T07", "Add conversation memory", 1, "T04", "Remember questions within a session."],
    ["T08", "Evaluate answers", 2, "T04", "Test five questions and citation correctness."],
    ["T09", "Prepare demo", 1, "T08", "Prepare and rehearse three demo questions."],
    ["T10", "Deploy application", 3, "T05,T08", "Publish interface and configure API secrets."],
]

with open("project_tasks.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow([
        "task_id", "task", "hours", "depends_on",
        "description", "estimate_basis"
    ])
    for task in tasks:
        writer.writerow(task + ["Synthetic demo estimate in person-hours"])

print("Created project_tasks.csv with 10 tasks")

Created project_tasks.csv with 10 tasks


In [ ]:
endgame_csv_docs = CSVLoader(
    file_path="project_tasks.csv",
    source_column="task_id"
).load()

# Spliting documents into chunks
This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

help(RecursiveCharacterTextSplitter)

Help on class RecursiveCharacterTextSplitter in module langchain_text_splitters.character:

class RecursiveCharacterTextSplitter(langchain_text_splitters.base.TextSplitter)
 |  RecursiveCharacterTextSplitter(
 |      separators: 'list[str] | None' = None,
 |      keep_separator: "bool | Literal['start', 'end']" = True,
 |      is_separator_regex: 'bool' = False,
 |      **kwargs: 'Any'
 |  ) -> 'None'
 |
 |  Splitting text by recursively look at characters.
 |
 |  Recursively tries to split by different characters to find one
 |  that works.
 |
 |  Method resolution order:
 |      RecursiveCharacterTextSplitter
 |      langchain_text_splitters.base.TextSplitter
 |      langchain_core.documents.transformers.BaseDocumentTransformer
 |      abc.ABC
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      separators: 'list[str] | None' = None,
 |      keep_separator: "bool | Literal['start', 'end']" = True,
 |      is_separator_regex: 'bool' = False,
 |  

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    length_function=len,
    is_separator_regex=False,
    separators=["\n\n", "\n", " ", ""],
)

In [ ]:
chunked_endgame_wikipedia_docs = text_splitter.split_documents(endgame_wikipedia_docs)
chunked_endgame_csv_docs = text_splitter.split_documents(endgame_csv_docs)
print(
    f"Number of documents in chunked_endgame_wikipedia_docs: {len(chunked_endgame_wikipedia_docs)}"
)
print(
    f"Number of documents in chunked_endgame_csv_docs: {len(chunked_endgame_csv_docs)}"
)

Number of documents in chunked_endgame_wikipedia_docs: 39
Number of documents in chunked_endgame_csv_docs: 10


# Embedding and Setting Up Storage With FAISS

In [ ]:
from langchain_openai import OpenAIEmbeddings

# https://api.python.langchain.com/en/latest/storage/langchain.storage.file_system.LocalFileStore.html#
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings  # https://python.langchain.com/docs/modules/data_connection/text_embedding/caching_embeddings


embeddings_model = OpenAIEmbeddings(model="text-embedding-ada-002")

# Instantiate the LocalFileStore with the local path
file_store = LocalFileStore("./cache/")

embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model,
    file_store,
    namespace=embeddings_model.model
)

In [ ]:
# Create the vector store with documents:
from langchain_community.vectorstores import FAISS

endgame_csv_db = FAISS.from_documents(chunked_endgame_csv_docs, embedder)
endgame_wiki_db = FAISS.from_documents(chunked_endgame_wikipedia_docs, embedder)

In [ ]:
# Set up FAISS retriever:
endgame_csv_faiss_retriever = endgame_csv_db.as_retriever(search_kwargs={"k": 4})
endgame_wiki_faiss_retriever = endgame_wiki_db.as_retriever(search_kwargs={"k": 4})

# Multiple retrievers
In this lab, we will use two retrievers and two sources
1. FAISS retriver
2. BM25 Retriever, a popular ranking function used in information retrieval systems to estimate the relevance of documents to a given search query
    https://python.langchain.com/docs/integrations/retrievers/bm25

Then `Ensemble Retriever` will be used to rerank the results of multiple retrevers in order to achieve better performance.
Sometimes you may want to retrieve documents from multiple different sources, or using multiple different algorithms. The ensemble retriever allows you to easily do this.
https://python.langchain.com/docs/modules/data_connection/retrievers/ensemble

In [ ]:
# BM25 retriever:
from langchain_community.retrievers import BM25Retriever

endgame_csv_bm25_retriever = BM25Retriever.from_documents(chunked_endgame_csv_docs, k=4)
endgame_wiki_bm25_retriever = BM25Retriever.from_documents(
    chunked_endgame_wikipedia_docs, k=4
)

In [ ]:
# Ensemble retriever with BM25 and FAISS
from langchain_classic.retrievers import EnsembleRetriever

endgame_ensemble_retriever = EnsembleRetriever(
    retrievers=[
        endgame_csv_bm25_retriever,
        endgame_wiki_bm25_retriever,
        endgame_csv_faiss_retriever,
        endgame_wiki_faiss_retriever,
    ],
    weights=[0.25, 0.25, 0.25, 0.25],
)

In [ ]:
docs = endgame_ensemble_retriever.invoke("Evaluate answers for a PDF assistant: test questions and citation correctness.")
docs

[Document(metadata={'source': 'T08', 'row': 7}, page_content='task_id: T08\ntask: Evaluate answers\nhours: 2\ndepends_on: T04\ndescription: Test five questions and citation correctness.\nestimate_basis: Synthetic demo estimate in person-hours'),
 Document(metadata={'source': 'T02', 'row': 1}, page_content='task_id: T02\ntask: Extract PDF text\nhours: 2\ndepends_on: T01\ndescription: Read a text-based PDF without OCR.\nestimate_basis: Synthetic demo estimate in person-hours'),
 Document(id='839882d3-f467-4c11-889f-eccc968abac0', metadata={'title': 'Minimum viable product', 'summary': "A minimum viable product (MVP) is a version of a product with just enough features to be usable by early customers who can then provide feedback for future product development.\nA focus on releasing an MVP means that developers potentially avoid lengthy and (possibly) unnecessary work. Instead, they iterate on working versions and respond to feedback, challenging and validating assumptions about a product'

In [ ]:
docs = endgame_ensemble_retriever.invoke(
    "Users should be able to ask follow-up questions without repeating what they said earlier."
)
docs

[Document(metadata={'source': 'T07', 'row': 6}, page_content='task_id: T07\ntask: Add conversation memory\nhours: 1\ndepends_on: T04\ndescription: Remember questions within a session.\nestimate_basis: Synthetic demo estimate in person-hours'),
 Document(metadata={'source': 'T08', 'row': 7}, page_content='task_id: T08\ntask: Evaluate answers\nhours: 2\ndepends_on: T04\ndescription: Test five questions and citation correctness.\nestimate_basis: Synthetic demo estimate in person-hours'),
 Document(metadata={'title': 'Software prototyping', 'summary': 'Software prototyping is the activity of creating prototypes of software applications, i.e., incomplete versions of the software program being developed. It is an activity that can occur in software development and is comparable to prototyping as known from other fields, such as mechanical engineering or manufacturing.\nA prototype typically simulates only a few aspects of, and may be highly different from, the final product. \nPrototyping ha

# Agents
The core idea of agents is to use a language model to choose a sequence of actions to take. In chains, a sequence of actions is hardcoded (in code). In agents, a language model is used as a reasoning engine to determine which actions to take and in which order.

reference: https://python.langchain.com/docs/modules/agents/

In [ ]:
# Tavily(https://tavily.com/) can be used as a search engine to get information online. You have to register to get an API key.
tavily_api_key = getpass.getpass("Enter your Tavily API Key: ")
os.environ["TAVILY_API_KEY"] = tavily_api_key

Enter your Tavily API Key: ··········


In [ ]:
# https://docs.tavily.com/docs/tavily-api/langchain
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

search = TavilySearchAPIWrapper()
tavily_tool = TavilySearchResults(api_wrapper=search, max_results=3)
tavily_tool.run("Streamlit official documentation create a simple app")

[{'title': "Building Your First Streamlit Application: A Beginner's Guide",
  'url': 'https://dev.to/shaheryaryousaf/building-your-first-streamlit-application-a-beginners-guide-14ki',
  'content': '## Setting Up Your Environment\n\nFirst, you need to install Streamlit. It\'s as simple as running this command in your Python environment:\n\n```\npip install streamlit \n```\n\n## Creating a Simple Streamlit App\n\nLet’s create a basic app that takes user input and displays it. Open your favorite IDE or text editor, create a new Python file (app.py), and let\'s start coding.\n\n### Step 1: Import Streamlit\n\n```\nimport streamlit as st \n```\n\n### Step 2: Add Title and User Input\n\nStreamlit makes it extremely easy to add elements to your app. Let\'s add a title and a text input field:\n\n```\nst.title(\'My First Streamlit App\') user_input = st.text_input("Enter some text") st.write(\'The user entered:\', user_input) \n```\n\n### Step 3: Run Your App [...] ```\nimport pandas as pd impo

In [ ]:
# https://python.langchain.com/docs/modules/agents/quick_start
from langchain_core.tools.retriever import create_retriever_tool

endgame_docs_retrieval_tool = create_retriever_tool(
    retriever=endgame_ensemble_retriever,
    name="project_rescue_search",
    description=(
        "Search project planning knowledge and a task catalog for building "
        "a PDF question-answering assistant. Use for MVP scope, prototyping, "
        "task IDs, estimated person-hours, and task dependencies. "
        "Time estimates are synthetic demo data. "
        "Use English search queries because the documents are in English."
    ),
)

endgame_retriever_tools = [
    endgame_docs_retrieval_tool,
    tavily_tool,
]

Tools have been created so we are ready to build an agent with them.

In [ ]:
import csv
from langchain_core.tools import tool

with open("project_tasks.csv", encoding="utf-8", newline="") as file:
    task_catalog = {
        row["task_id"]: row
        for row in csv.DictReader(file)
    }

@tool
def check_project_plan(task_ids: list[str], available_hours: float) -> dict:
    """Check selected task IDs against the CSV catalog.
    Return total person-hours, budget fit, and missing prerequisites.
    Estimates are synthetic demo data, not guaranteed completion times.
    """
    if available_hours < 0:
        return {"error": "Available hours must be non-negative."}

    selected = list(dict.fromkeys(
        task_id.strip().upper() for task_id in task_ids
    ))

    if not selected:
        return {"error": "Select at least one task."}

    unknown = [
        task_id for task_id in selected
        if task_id not in task_catalog
    ]
    if unknown:
        return {"error": "Unknown task IDs.", "unknown_ids": unknown}

    total = sum(
        float(task_catalog[task_id]["hours"])
        for task_id in selected
    )

    missing = {}
    for task_id in selected:
        dependencies = [
            item.strip()
            for item in task_catalog[task_id]["depends_on"].split(",")
            if item.strip() and item.strip().lower() != "none"
        ]
        absent = [item for item in dependencies if item not in selected]
        if absent:
            missing[task_id] = absent

    return {
        "selected_tasks": selected,
        "total_person_hours": total,
        "available_person_hours": available_hours,
        "within_budget": total <= available_hours,
        "over_budget_by": max(0, total - available_hours),
        "missing_prerequisites": missing,
        "passes_budget_and_dependency_checks": (
            total <= available_hours and not missing
        ),
        "estimate_basis": "Synthetic demo estimates in person-hours",
    }

print(check_project_plan.invoke({
    "task_ids": ["T01", "T02", "T03", "T04", "T08", "T07", "T09", "T10"],
    "available_hours": 12,
}))

{'selected_tasks': ['T01', 'T02', 'T03', 'T04', 'T08', 'T07', 'T09', 'T10'], 'total_person_hours': 15.0, 'available_person_hours': 12.0, 'within_budget': False, 'over_budget_by': 3.0, 'missing_prerequisites': {'T10': ['T05']}, 'passes_budget_and_dependency_checks': False, 'estimate_basis': 'Synthetic demo estimates in person-hours'}


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

endgame_retriever_tools = [
    endgame_docs_retrieval_tool,
    tavily_tool,
    check_project_plan,
]

endgame_agent_executor = create_react_agent(
    llm,
    endgame_retriever_tools,
    prompt=(
        "You are Project Rescue, a planning assistant for PDF QA demos. "
        "For every proposed plan, call check_project_plan before answering. "
        "If a plan exceeds the budget or has missing prerequisites, revise "
        "it and call check_project_plan again on the revised task IDs. "
        "Never present an unchecked or failed plan as valid. "
        "Use the checker's total hours, not your own calculation. "
        "Search results are partial: never conclude that a task does not "
        "exist merely because retrieval did not return it. "
        "Preserve the user's required deliverables and their prerequisites. "
        "If you cannot find a valid plan, explain the limitation honestly. "
        "Task-hour estimates are synthetic demo data. "
        "Use conversation history for user-provided names and budgets."
    ),
)

/tmp/ipykernel_1624/2936846200.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  endgame_agent_executor = create_react_agent(


In [ ]:
# Use .invoke() instead of calling directly
response = endgame_agent_executor.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
    "Use project_rescue_search to look up task T08. "
    "What is its purpose, estimated person-hours, "
    "and prerequisite task? "
    "Mention that the estimate is synthetic demo data."
                )
            )
        ]
    }
)


print(response["messages"][-1].content)

Task T08 is to "Evaluate answers" with an estimated duration of 2 person-hours. It depends on the prerequisite task T04, which involves testing five questions and citation correctness. The estimate provided is synthetic demo data.


In [ ]:
# https://python.langchain.com/docs/expression_language/how_to/message_history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnableLambda

message_history = {}

def get_session_history(session_id):
    if session_id not in message_history:
        message_history[session_id] = ChatMessageHistory()
    return message_history[session_id]

def prepare_agent_input(data):
    return {
        "messages": list(data.get("chat_history", [])) + list(data["messages"])
    }

def prepare_history_output(result):
    return {
        **result,
        "output": result["messages"][-1],
    }
# إعادة تشغيل هذه الخلية تصفّر السجل؛ .
endgame_agent_with_chat_history = RunnableWithMessageHistory(
    runnable=(
        RunnableLambda(prepare_agent_input)
        | endgame_agent_executor
        | RunnableLambda(prepare_history_output)
    ),
    get_session_history=get_session_history,
    input_messages_key="messages",
    history_messages_key="chat_history",
    output_messages_key="output",
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"session_id": "1"}}

response = endgame_agent_with_chat_history.invoke(
    {"messages": [HumanMessage(content=
                               "My project is called RescuePDF. "
                               "I have 12 person-hours to build a PDF question-answering assistant. "
                               "Confirm only the project name and available person-hours. Do not propose tasks or use tools.")]},
    config=config
)

print(response["messages"][-1].content)

The project name is RescuePDF, and you have 12 person-hours available for building the PDF question-answering assistant.


In [ ]:
from langchain_core.messages import HumanMessage

response = endgame_agent_with_chat_history.invoke(
    {"messages": [HumanMessage(content=
                    "What is my project called, and how many person-hours "
                    "did I say I have? Answer only these two points.")]},
    config={"configurable": {"session_id": "1"}}
)

print(response["messages"][-1].content)

Your project is called RescuePDF, and you mentioned that you have 12 person-hours available.


In [ ]:
config = {"configurable": {"session_id": "1"}}

response = endgame_agent_with_chat_history.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "For the same project and person-hour budget I mentioned earlier, "
                    "my proposed tasks are "
                    "T01, T02, T03, T04, T08, T07, T09, T10. "
                    "Use your tools to check this plan. "
                    "If it fails, revise it while keeping PDF answers with "
                    "sources, evaluation, and demo preparation. "
                    "Check the revised plan before presenting it. "
                    "Report task IDs, total hours, and what you removed. "
                    "State that the hours are synthetic demo estimates."
                )
            )
        ]
    },
    config=config,
)

print(response["messages"][-1].content)

for message in response["messages"]:
    for call in getattr(message, "tool_calls", []):
        print("Tool used:", call["name"])

After revising the plan, here is the validated task list for your project RescuePDF with 12 person-hours:

- Task IDs: T01, T02, T03, T04, T08, T07, T09
- Total Hours: 12.0 hours
- Removed Task: T10 (due to missing prerequisite T05)

The hours provided are synthetic demo estimates.
Tool used: check_project_plan
Tool used: check_project_plan
